In [0]:
# ==========================================
# 1. WARSTWA GOLD - Agregacje biznesowe
# ==========================================
print("⏳ Budowanie tabeli Gold (Gotowej do podpięcia pod Power BI)...")
spark.sql("""
CREATE OR REPLACE TABLE dbw_showcase.default.payroll_gold AS
SELECT 
    h.department_name,
    h.city,
    COUNT(p.transaction_id) AS total_transactions,
    ROUND(SUM(p.hours_logged), 2) AS total_hours
FROM dbw_showcase.default.payroll_silver p
JOIN dbw_showcase.default.hr_silver h 
  ON p.emp_id = h.emp_id
WHERE h.is_current = true
GROUP BY h.department_name, h.city
""")
print("✅ Tabela Gold utworzona pomyślnie!")

# ==========================================
# 2. DELTA MAINTENANCE: OPTIMIZE & Z-ORDER
# ==========================================
print("⏳ Optymalizacja fizycznego składowania danych (Small File Problem)...")
spark.sql("OPTIMIZE dbw_showcase.default.payroll_silver ZORDER BY (emp_id)")
print("✅ Tabela Silver zoptymalizowana (OPTIMIZE + Z-ORDER)!")

# ==========================================
# 3. GDPR COMPLIANCE: Prawo do bycia zapomnianym (DELETE + VACUUM)
# ==========================================
print("⏳ Wykonywanie usunięcia (GDPR)...")
# Krok A: Usuwamy pracownika z logiki biznesowej
spark.sql("DELETE FROM dbw_showcase.default.hr_silver WHERE emp_id = 10102")

# Krok B: Wykonujemy standardowy VACUUM
# UWAGA NA REKRUTACJĘ: Architektura Serverless blokuje wyłączenie 'retentionDurationCheck'. 
# Polegamy na domyślnej 7-dniowej retencji Unity Catalog, aby chronić zapytania współbieżne.
spark.sql("VACUUM dbw_showcase.default.hr_silver")

print("✅ Proces GDPR zrealizowany (fizyczne usunięcie nastąpi zgodnie z bezpieczną polityką retencji)!")